In [ ]:
# ==============================================================================
# BTK DATATHON 2026 | career_success_score Regression
# Metric: MSE  |  Method: LightGBM + XGBoost + CatBoost Blend
# Install:  pip install lightgbm xgboost catboost scikit-learn scipy pandas numpy
# ==============================================================================

import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# -----------------------------------------------------------------------
# CONFIG  (adjust paths if needed)
# -----------------------------------------------------------------------
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
SUB_PATH   = "submission.csv"
N_FOLDS    = 5
SEED       = 42
np.random.seed(SEED)

# -----------------------------------------------------------------------
# 1. LOAD
# -----------------------------------------------------------------------
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

TARGET   = "career_success_score"
ID_COL   = "student_id"
TEXT_COL = "mentor_feedback_text"

test_ids = test[ID_COL].values
y        = train[TARGET].values
n_train  = len(train)

print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Target  -->  mean={y.mean():.2f}  std={y.std():.2f}  "
      f"min={y.min():.2f}  max={y.max():.2f}\n")

# -----------------------------------------------------------------------
# 2. COMBINED FEATURE ENGINEERING  (train + test together to avoid leakage)
# -----------------------------------------------------------------------
combined = pd.concat(
    [train.drop(columns=[TARGET, ID_COL]),
     test.drop(columns=[ID_COL])],
    axis=0, ignore_index=True
)

# Semantic zero-fills for absent activity columns
ZERO_FILL = ["github_avg_stars", "open_source_contribution_count",
             "internship_duration_months"]
for col in ZERO_FILL:
    combined[col] = combined[col].fillna(0)

# -------  2a. NLP: TF-IDF + Truncated SVD on Turkish mentor text  -------
combined[TEXT_COL] = combined[TEXT_COL].fillna("")

tfidf   = TfidfVectorizer(max_features=300, ngram_range=(1, 2),
                           min_df=3, sublinear_tf=True, strip_accents="unicode")
T_mat   = tfidf.fit_transform(combined[TEXT_COL])
svd     = TruncatedSVD(n_components=30, random_state=SEED)
T_svd   = svd.fit_transform(T_mat)
text_df = pd.DataFrame(T_svd, columns=[f"txt_{i}" for i in range(30)])

# Surface-level text statistics
combined["txt_len"]   = combined[TEXT_COL].str.len().fillna(0)
combined["txt_words"] = combined[TEXT_COL].str.split().str.len().fillna(0)
combined["txt_uniq"]  = combined[TEXT_COL].str.split().apply(
    lambda x: len(set(x)) if isinstance(x, list) else 0)

# Turkish sentiment keyword signals
POS_KW = ["guclu", "basarili", "etkileyici", "potansiyel", "ozveri",
           "yetenekli", "olagan", "mukemmel", "umut", "katki", "parlak",
           "dikkat cekici", "gozlemledim"]
NEG_KW = ["gelistirmeli", "eksik", "zayif", "yetersiz", "zorlan",
           "acemi", "gelistirme", "calisma gerek", "acikar"]

# Normalize accented Turkish for simple matching
def _deaccent(s):
    return (s.lower()
             .replace("ü","u").replace("ö","o").replace("ı","i").replace("ğ","g")
             .replace("ş","s").replace("ç","c"))

combined["pos_kw"]     = combined[TEXT_COL].apply(lambda x: sum(kw in _deaccent(x) for kw in POS_KW))
combined["neg_kw"]     = combined[TEXT_COL].apply(lambda x: sum(kw in _deaccent(x) for kw in NEG_KW))
combined["sent_ratio"] = (combined["pos_kw"] + 1) / (combined["neg_kw"] + 1)

# -------  2b. Domain composite features  -------
TECH_COLS = [
    "coding_score", "problem_solving_score", "data_structures_score",
    "sql_score", "machine_learning_score", "backend_score",
    "frontend_score", "cloud_score", "devops_score"
]
SOCIAL_COLS = [
    "communication_score", "teamwork_score",
    "leadership_score", "presentation_score"
]

combined["tech_mean"] = combined[TECH_COLS].mean(axis=1)
combined["tech_max"]  = combined[TECH_COLS].max(axis=1)
combined["tech_min"]  = combined[TECH_COLS].min(axis=1)
combined["tech_std"]  = combined[TECH_COLS].std(axis=1)
combined["tech_sum"]  = combined[TECH_COLS].sum(axis=1)
combined["tech_top3"] = combined[TECH_COLS].apply(
    lambda r: r.nlargest(3).mean(), axis=1)

combined["social_mean"] = combined[SOCIAL_COLS].mean(axis=1)
combined["social_max"]  = combined[SOCIAL_COLS].max(axis=1)
combined["social_sum"]  = combined[SOCIAL_COLS].sum(axis=1)

combined["interview_composite"] = (
    combined["technical_interview_score"].fillna(combined["technical_interview_score"].median()) +
    combined["hr_interview_score"].fillna(combined["hr_interview_score"].median())
) / 2

combined["experience_index"] = (
    combined["internship_count"] * 3.0 +
    combined["internship_duration_months"] * 0.5 +
    combined["real_client_project_count"] * 2.0 +
    combined["freelance_project_count"] * 1.0 +
    combined["hackathon_count"] * 1.5 +
    combined["hackathon_awards"] * 2.5 +
    combined["open_source_contribution_count"] * 0.5
)

combined["portfolio_composite"] = (
    combined["portfolio_score"].fillna(combined["portfolio_score"].median()) +
    combined["github_repo_count"] * 0.3 +
    combined["github_avg_stars"] * 0.5 +
    combined["linkedin_profile_score"].fillna(combined["linkedin_profile_score"].median()) * 0.5 +
    combined["cv_quality_score"] * 0.5
)

combined["academic_composite"] = (
    combined["cgpa"] * 10 +
    combined["english_exam_score"].fillna(combined["english_exam_score"].median()) * 0.3 +
    combined["attendance_rate"] * 0.2 -
    combined["failed_courses_count"] * 3.0
)

combined["app_efficiency"] = np.where(
    combined["applications_sent"] > 0,
    combined["interviews_attended"] / combined["applications_sent"],
    0.0
)
combined["years_since_grad"] = combined["application_year"] - combined["graduation_year"]
combined["learning_count"]   = combined["certification_count"] + combined["bootcamp_count"]

# First-order interaction features
combined["tech_x_interview"]   = combined["tech_mean"] * combined["interview_composite"] / 100
combined["social_x_interview"] = combined["social_mean"] * combined["interview_composite"] / 100
combined["cgpa_x_tech"]        = combined["cgpa"] * combined["tech_mean"] / 4.0
combined["portfolio_x_exp"]    = combined["portfolio_composite"] * (combined["experience_index"] + 1)
combined["proj_x_qual"]        = (combined["real_client_project_count"] *
                                   combined["project_quality_score"])
combined["hack_x_award"]       = (combined["hackathon_count"] *
                                   (combined["hackathon_awards"] + 1))

# -------  2c. Label-encode categoricals  -------
CAT_COLS = [
    "department", "university_tier", "target_role",
    "hobby", "preferred_social_media_platform"
]
for col in CAT_COLS:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# -------  2d. Assemble final feature matrix  -------
combined.drop(columns=[TEXT_COL], inplace=True)
combined = pd.concat(
    [combined.reset_index(drop=True), text_df.reset_index(drop=True)],
    axis=1
)
# Final pass: fill any residual NaN with column median
combined.fillna(combined.median(numeric_only=True), inplace=True)

X      = combined.iloc[:n_train].values.astype(np.float32)
X_test = combined.iloc[n_train:].values.astype(np.float32)

print(f"Feature count: {X.shape[1]}")
print(f"X: {X.shape}  |  X_test: {X_test.shape}\n")

# -----------------------------------------------------------------------
# 3. CROSS-VALIDATION  (5-Fold, OOF predictions)
# -----------------------------------------------------------------------
kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = {k: np.zeros(n_train)     for k in ["lgb", "xgb", "cat"]}
prd = {k: np.zeros(len(X_test)) for k in ["lgb", "xgb", "cat"]}
fold_rows = []

SEP = "=" * 68
print(SEP)
print(f"{'Fold':>5}  {'LGB_MSE':>11}  {'XGB_MSE':>11}  {'CAT_MSE':>11}  {'Avg_MSE':>11}")
print(SEP)

for fold, (tr_idx, vl_idx) in enumerate(kf.split(X, y), 1):
    Xtr, Xvl = X[tr_idx], X[vl_idx]
    ytr, yvl = y[tr_idx], y[vl_idx]

    # LightGBM
    m_lgb = lgb.LGBMRegressor(
        n_estimators=3000,     learning_rate=0.02,
        max_depth=7,           num_leaves=64,
        min_child_samples=20,  subsample=0.8,
        colsample_bytree=0.75, reg_alpha=0.05,
        reg_lambda=0.5,        random_state=SEED,
        n_jobs=-1,             verbose=-1
    )
    m_lgb.fit(
        Xtr, ytr, eval_set=[(Xvl, yvl)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]
    )
    oof["lgb"][vl_idx] = m_lgb.predict(Xvl)
    prd["lgb"] += m_lgb.predict(X_test) / N_FOLDS

    # XGBoost
    m_xgb = xgb.XGBRegressor(
        n_estimators=3000,     learning_rate=0.02,
        max_depth=6,           subsample=0.8,
        colsample_bytree=0.75, reg_alpha=0.05,
        reg_lambda=1.0,        random_state=SEED,
        n_jobs=-1,             verbosity=0,
        early_stopping_rounds=150, eval_metric="rmse"
    )
    m_xgb.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
    oof["xgb"][vl_idx] = m_xgb.predict(Xvl)
    prd["xgb"] += m_xgb.predict(X_test) / N_FOLDS

    # CatBoost
    m_cat = CatBoostRegressor(
        iterations=3000, learning_rate=0.02, depth=7,
        l2_leaf_reg=3,   random_seed=SEED,
        early_stopping_rounds=150, verbose=0
    )
    m_cat.fit(Xtr, ytr, eval_set=(Xvl, yvl))
    oof["cat"][vl_idx] = m_cat.predict(Xvl)
    prd["cat"] += m_cat.predict(X_test) / N_FOLDS

    ml = mean_squared_error(yvl, oof["lgb"][vl_idx])
    mx = mean_squared_error(yvl, oof["xgb"][vl_idx])
    mc = mean_squared_error(yvl, oof["cat"][vl_idx])
    fold_rows.append({"fold": fold, "lgb": ml, "xgb": mx, "cat": mc,
                       "avg": (ml + mx + mc) / 3})
    print(f"{fold:>5}  {ml:>11.4f}  {mx:>11.4f}  {mc:>11.4f}  {(ml+mx+mc)/3:>11.4f}")

print(SEP)
avgs = {k: np.mean([r[k] for r in fold_rows]) for k in ["lgb", "xgb", "cat", "avg"]}
stds = {k: np.std( [r[k] for r in fold_rows]) for k in ["lgb", "xgb", "cat", "avg"]}
print(f"{'Mean':>5}  {avgs['lgb']:>11.4f}  {avgs['xgb']:>11.4f}  {avgs['cat']:>11.4f}  {avgs['avg']:>11.4f}")
print(f"{'Std':>5}  {stds['lgb']:>11.4f}  {stds['xgb']:>11.4f}  {stds['cat']:>11.4f}  {stds['avg']:>11.4f}")
print(SEP)

# -----------------------------------------------------------------------
# 4. OPTIMAL BLEND  (Nelder-Mead minimizing OOF MSE)
# -----------------------------------------------------------------------
def _blend_loss(w):
    w = np.abs(w); w /= w.sum()
    blend = w[0] * oof["lgb"] + w[1] * oof["xgb"] + w[2] * oof["cat"]
    return mean_squared_error(y, blend)

opt = minimize(_blend_loss, [1/3, 1/3, 1/3], method="Nelder-Mead",
                options={"xatol": 1e-7, "fatol": 1e-7, "maxiter": 30000})
W = np.abs(opt.x); W /= W.sum()

oof_blend = W[0]*oof["lgb"] + W[1]*oof["xgb"] + W[2]*oof["cat"]
prd_blend = W[0]*prd["lgb"] + W[1]*prd["xgb"] + W[2]*prd["cat"]
prd_final = np.clip(prd_blend, 0, 100)

# -----------------------------------------------------------------------
# 5. DETAILED METRICS REPORT
# -----------------------------------------------------------------------
SEP2 = "=" * 76
print(f"\n{SEP2}")
print("FINAL OOF METRICS  (5-Fold Cross-Validation  |  Not yet submitted)")
print(SEP2)
print(f"{'Model':<14}  {'MSE':>9}  {'RMSE':>9}  {'MAE':>9}  {'R2':>9}  {'MaxAbsErr':>11}")
print("-" * 76)
for name, arr in [("LightGBM", oof["lgb"]),
                   ("XGBoost",  oof["xgb"]),
                   ("CatBoost", oof["cat"]),
                   ("BLEND",    oof_blend)]:
    mse  = mean_squared_error(y, arr)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y, arr)
    r2   = r2_score(y, arr)
    maxe = np.abs(y - arr).max()
    flag = "  <-- submission" if name == "BLEND" else ""
    print(f"{name:<14}  {mse:>9.4f}  {rmse:>9.4f}  {mae:>9.4f}  {r2:>9.4f}  {maxe:>11.4f}{flag}")
print(SEP2)

print(f"\nBlend weights    :  LGB = {W[0]:.4f}  |  XGB = {W[1]:.4f}  |  CAT = {W[2]:.4f}")
print(f"True  mean / std :  {y.mean():.4f}  /  {y.std():.4f}")
print(f"OOF   mean / std :  {oof_blend.mean():.4f}  /  {oof_blend.std():.4f}")
print(f"Test  pred range :  [{prd_final.min():.2f},  {prd_final.max():.2f}]")

residuals = y - oof_blend
print(f"\nAbsolute error coverage (OOF):")
for thr in [3, 5, 8, 10, 15, 20]:
    frac = (np.abs(residuals) <= thr).mean() * 100
    print(f"  |error| <= {thr:>2}  :  {frac:>5.1f}%  "
          f"({int(frac * n_train / 100):>5} / {n_train} samples)")

pcts = [5, 10, 25, 50, 75, 90, 95]
pct_str = "  |  ".join(f"P{p:02d} = {np.percentile(residuals, p):+6.2f}" for p in pcts)
print(f"\nResidual percentiles (true minus pred):")
print(f"  {pct_str}")

print(f"\nPer-fold MSE breakdown:")
print(f"  {'Fold':>5}  {'LGB':>9}  {'XGB':>9}  {'CAT':>9}  {'Avg':>9}")
print(f"  {'':->50}")
for r in fold_rows:
    print(f"  {r['fold']:>5}  {r['lgb']:>9.4f}  {r['xgb']:>9.4f}  {r['cat']:>9.4f}  {r['avg']:>9.4f}")
print(f"  {'Mean':>5}  {avgs['lgb']:>9.4f}  {avgs['xgb']:>9.4f}  {avgs['cat']:>9.4f}  {avgs['avg']:>9.4f}")
print(f"  {'Std':>5}  {stds['lgb']:>9.4f}  {stds['xgb']:>9.4f}  {stds['cat']:>9.4f}  {stds['avg']:>9.4f}")

# -----------------------------------------------------------------------
# 6. SAVE SUBMISSION
# -----------------------------------------------------------------------
submission = pd.DataFrame({ID_COL: test_ids, TARGET: prd_final})
submission.to_csv(SUB_PATH, index=False)
print(f"\nSubmission  -->  '{SUB_PATH}'  ({len(submission)} rows)")
print(submission.head(5).to_string(index=False))